# Chapter 6.5: FP8 & Quantization Support in SGLang

## Learning Objectives

By the end of this notebook, you will:

1. **Understand FP8 format** variants: E4M3 and E5M2
2. **Compare online vs offline** FP8 quantization approaches
3. **Learn per-tensor vs per-channel** FP8 scaling strategies
4. **Explore FP8 KV-cache quantization** for memory savings
5. **Understand W8A8 and W4A16** quantization methods
6. **Integrate with popular quantization** frameworks (GPTQ, AWQ, SqueezeLLM)
7. **Analyze performance benchmarks** for different quantization methods
8. **Walk through SGLang source code** for quantization support

## Prerequisites

- Understanding of floating-point number representation
- Basic knowledge of linear algebra operations in LLMs
- Familiarity with GPU memory constraints

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part6/chapter_6.5_fp8_quantization.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part6/chapter_6.5_fp8_quantization.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Why Quantization for LLM Inference?

### 1.1 The Memory Problem

LLM inference is memory-bound. A 70B parameter model in FP16 requires:
- **Model weights**: 70B * 2 bytes = 140 GB
- **KV-cache**: ~2-4 GB per concurrent request (depending on context length)
- **Total**: Needs multiple A100-80GB GPUs

Quantization reduces memory by representing values with fewer bits:

| Format | Bits | 70B Model Size | Memory Savings |
|--------|------|---------------|----------------|
| FP32 | 32 | 280 GB | Baseline |
| FP16/BF16 | 16 | 140 GB | 2x |
| FP8 | 8 | 70 GB | 4x |
| INT8 | 8 | 70 GB | 4x |
| INT4 | 4 | 35 GB | 8x |

### 1.2 Quantization Approaches

```
+-------------------------------------------+
|         Quantization Taxonomy             |
+-------------------------------------------+
|                                           |
|  Weight-only Quantization                 |
|    W4A16: 4-bit weights, FP16 activations |
|    W8A16: 8-bit weights, FP16 activations |
|    Examples: GPTQ, AWQ, SqueezeLLM       |
|                                           |
|  Weight + Activation Quantization         |
|    W8A8: 8-bit weights and activations    |
|    FP8: E4M3 for weights, E5M2 for grads |
|    Examples: SmoothQuant, FP8 native      |
|                                           |
|  KV-Cache Quantization                    |
|    FP8 KV-cache: 2x memory savings       |
|    INT8 KV-cache                          |
+-------------------------------------------+
```

In [ ]:
import struct
import numpy as np

# Understand floating-point formats

def format_info(name, sign_bits, exp_bits, mantissa_bits):
    """Display floating-point format characteristics."""
    total_bits = sign_bits + exp_bits + mantissa_bits
    bias = (2 ** (exp_bits - 1)) - 1
    max_exp = 2 ** exp_bits - 1 - bias
    min_exp = 1 - bias
    max_val = (2 - 2**(-mantissa_bits)) * (2 ** max_exp)
    min_normal = 2 ** min_exp
    precision = 2 ** (-mantissa_bits)
    
    return {
        "name": name,
        "total_bits": total_bits,
        "sign": sign_bits,
        "exponent": exp_bits,
        "mantissa": mantissa_bits,
        "bias": bias,
        "max_value": max_val,
        "min_normal": min_normal,
        "relative_precision": precision,
    }

formats = [
    format_info("FP32",        1, 8, 23),
    format_info("FP16",        1, 5, 10),
    format_info("BF16",        1, 8, 7),
    format_info("FP8 E4M3",   1, 4, 3),
    format_info("FP8 E5M2",   1, 5, 2),
]

print("=== Floating-Point Format Comparison ===")
print(f"{'Format':>12s} | {'Bits':>5s} | {'S:E:M':>7s} | {'Max Value':>15s} | {'Precision':>12s}")
print("-" * 65)
for f in formats:
    print(f"{f['name']:>12s} | {f['total_bits']:>5d} | {f['sign']}:{f['exponent']}:{f['mantissa']:>2d}  | "
          f"{f['max_value']:>15.1f} | {f['relative_precision']:>12.6f}")

## 2. FP8 Format Deep Dive

### 2.1 E4M3 vs E5M2

FP8 has two variants defined by the IEEE working group:

```
FP8 E4M3 (4-bit exponent, 3-bit mantissa):
  [S][EEEE][MMM]
  - Range: [-448, 448]
  - Precision: ~1/8 of the value
  - Best for: Weights and activations (needs precision)

FP8 E5M2 (5-bit exponent, 2-bit mantissa):
  [S][EEEEE][MM]
  - Range: [-57344, 57344]
  - Precision: ~1/4 of the value
  - Best for: Gradients (needs dynamic range)
```

### 2.2 Why E4M3 for Inference?

For inference, we use **E4M3** because:
1. Weights have a known, bounded range (can be scaled to fit)
2. Precision matters more than range for quality
3. 3 mantissa bits give 8 discrete values per power-of-2 interval

### 2.3 Quantization Formula

```
Quantize:   x_fp8 = clamp(round(x_fp16 / scale), FP8_MIN, FP8_MAX)
Dequantize: x_fp16 = x_fp8 * scale

Where:
  scale = max(abs(x_fp16)) / FP8_MAX
```

In [ ]:
# Simulate FP8 quantization

class FP8Quantizer:
    """Simulates FP8 E4M3 quantization."""
    
    FP8_E4M3_MAX = 448.0
    FP8_E4M3_MIN = -448.0
    
    FP8_E5M2_MAX = 57344.0
    FP8_E5M2_MIN = -57344.0
    
    @staticmethod
    def quantize_per_tensor(tensor, format="e4m3"):
        """Per-tensor FP8 quantization."""
        if format == "e4m3":
            fp8_max = FP8Quantizer.FP8_E4M3_MAX
        else:
            fp8_max = FP8Quantizer.FP8_E5M2_MAX
        
        # Compute scale
        amax = np.max(np.abs(tensor))
        scale = amax / fp8_max if amax > 0 else 1.0
        
        # Quantize
        scaled = tensor / scale
        clipped = np.clip(scaled, -fp8_max, fp8_max)
        
        # Simulate reduced precision (round to nearest representable)
        if format == "e4m3":
            # 3 mantissa bits -> 8 values per power-of-2 interval
            n_mantissa_vals = 8
        else:
            n_mantissa_vals = 4
        
        # Simplified: round to nearest multiple of precision
        sign = np.sign(clipped)
        abs_val = np.abs(clipped)
        # Find the exponent
        exp = np.floor(np.log2(abs_val + 1e-10))
        # Quantize mantissa
        mantissa = abs_val / (2.0 ** exp)
        quantized_mantissa = np.round(mantissa * n_mantissa_vals) / n_mantissa_vals
        quantized = sign * quantized_mantissa * (2.0 ** exp)
        
        return quantized, scale
    
    @staticmethod
    def dequantize(quantized, scale):
        return quantized * scale
    
    @staticmethod
    def quantize_per_channel(tensor, axis=0, format="e4m3"):
        """Per-channel FP8 quantization (better precision)."""
        if format == "e4m3":
            fp8_max = FP8Quantizer.FP8_E4M3_MAX
        else:
            fp8_max = FP8Quantizer.FP8_E5M2_MAX
        
        # Compute scale per channel
        amax = np.max(np.abs(tensor), axis=1-axis, keepdims=True)
        scales = amax / fp8_max
        scales = np.maximum(scales, 1e-10)  # Avoid division by zero
        
        # Quantize
        scaled = tensor / scales
        clipped = np.clip(scaled, -fp8_max, fp8_max)
        
        return clipped, scales

# Demo: FP8 quantization quality
np.random.seed(42)

# Simulate model weights (normally distributed)
weights = np.random.randn(256, 512).astype(np.float32) * 0.02  # Typical weight scale

print("=== FP8 Quantization Quality Analysis ===")
print(f"Original weights: shape={weights.shape}, dtype=float32")
print(f"Weight stats: mean={weights.mean():.6f}, std={weights.std():.6f}, "
      f"min={weights.min():.6f}, max={weights.max():.6f}")

# Per-tensor quantization
q_tensor, scale_tensor = FP8Quantizer.quantize_per_tensor(weights, format="e4m3")
deq_tensor = FP8Quantizer.dequantize(q_tensor, scale_tensor)
error_tensor = np.abs(weights - deq_tensor)

# Per-channel quantization
q_channel, scale_channel = FP8Quantizer.quantize_per_channel(weights, axis=0, format="e4m3")
deq_channel = q_channel * scale_channel
error_channel = np.abs(weights - deq_channel)

print(f"\n--- Per-Tensor E4M3 ---")
print(f"Scale: {scale_tensor:.8f}")
print(f"Mean absolute error: {error_tensor.mean():.8f}")
print(f"Max absolute error:  {error_tensor.max():.8f}")
print(f"Relative error:      {(error_tensor / (np.abs(weights) + 1e-10)).mean():.4%}")

print(f"\n--- Per-Channel E4M3 ---")
print(f"Scales shape: {scale_channel.shape}")
print(f"Mean absolute error: {error_channel.mean():.8f}")
print(f"Max absolute error:  {error_channel.max():.8f}")
print(f"Relative error:      {(error_channel / (np.abs(weights) + 1e-10)).mean():.4%}")

In [ ]:
# Visualize quantization error distribution
try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ImportError:
    HAS_PLT = False

if HAS_PLT:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Original weight distribution
    axes[0].hist(weights.flatten(), bins=100, alpha=0.7, color='blue')
    axes[0].set_title('Original Weights (FP32)')
    axes[0].set_xlabel('Value')
    axes[0].set_ylabel('Count')
    
    # Per-tensor error
    axes[1].hist(error_tensor.flatten(), bins=100, alpha=0.7, color='red')
    axes[1].set_title('Per-Tensor Quantization Error')
    axes[1].set_xlabel('Absolute Error')
    
    # Per-channel error
    axes[2].hist(error_channel.flatten(), bins=100, alpha=0.7, color='green')
    axes[2].set_title('Per-Channel Quantization Error')
    axes[2].set_xlabel('Absolute Error')
    
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available.")

## 3. Online vs Offline FP8 Quantization

### 3.1 Offline (Static) Quantization

Quantize weights before deployment. Scales are computed once and stored.

```python
# Offline quantization workflow
# Step 1: Load FP16 model
model = load_model("llama-3-8b")

# Step 2: Calibrate scales using representative data
for batch in calibration_data:
    with torch.no_grad():
        model(batch)  # Collect activation statistics

# Step 3: Quantize and save
quantized_model = quantize_to_fp8(model, scales)
save_model(quantized_model, "llama-3-8b-fp8")
```

**Pros:** Better quality (calibrated scales), no runtime overhead
**Cons:** Requires calibration data, model-specific

### 3.2 Online (Dynamic) Quantization

Quantize activations on-the-fly during inference.

```python
# Online quantization during inference
def fp8_linear(x, weight_fp8, weight_scale):
    # Weights are pre-quantized (offline)
    # Activations are quantized dynamically
    x_scale = x.abs().max() / FP8_MAX
    x_fp8 = (x / x_scale).to(torch.float8_e4m3fn)
    
    # FP8 matmul
    output = torch._scaled_mm(x_fp8, weight_fp8, 
                               scale_a=x_scale, scale_b=weight_scale)
    return output
```

**Pros:** No calibration needed, works with any input
**Cons:** Scale computation adds overhead, may lose quality

### 3.3 Comparison

| Aspect | Offline | Online |
|--------|---------|--------|
| Setup | Calibration required | None |
| Quality | Better (tuned scales) | Good (dynamic scales) |
| Runtime overhead | None | Scale computation (~1%) |
| Weight storage | FP8 + scales | FP8 + scales |
| Activation handling | Pre-computed scales | Dynamic per-batch |

In [ ]:
# Simulate online vs offline quantization
import torch

class FP8LinearSimulation:
    """Simulate FP8 linear layer with different quantization modes."""
    
    def __init__(self, in_features, out_features):
        self.weight_fp32 = torch.randn(out_features, in_features) * 0.02
        self.fp8_max = 448.0
        
        # Offline: pre-compute weight scale
        self.weight_scale = self.weight_fp32.abs().max() / self.fp8_max
        self.weight_fp8 = torch.clamp(
            self.weight_fp32 / self.weight_scale, -self.fp8_max, self.fp8_max
        ).round()  # Simulate FP8 precision
    
    def forward_fp16(self, x):
        """Standard FP16 forward."""
        return x @ self.weight_fp32.T
    
    def forward_fp8_offline(self, x, activation_scale):
        """FP8 with offline (pre-calibrated) activation scale."""
        x_fp8 = torch.clamp(x / activation_scale, -self.fp8_max, self.fp8_max).round()
        # Simulate FP8 matmul
        result = x_fp8 @ self.weight_fp8.T
        # Dequantize
        return result * activation_scale * self.weight_scale
    
    def forward_fp8_online(self, x):
        """FP8 with online (dynamic) activation scale."""
        # Compute activation scale dynamically
        activation_scale = x.abs().max() / self.fp8_max
        return self.forward_fp8_offline(x, activation_scale)

# Test
torch.manual_seed(42)
layer = FP8LinearSimulation(512, 256)

# Generate test inputs with different distributions
test_inputs = {
    "Normal (std=1)": torch.randn(32, 512),
    "Normal (std=0.1)": torch.randn(32, 512) * 0.1,
    "Uniform [-1,1]": torch.rand(32, 512) * 2 - 1,
    "With outliers": torch.randn(32, 512) + torch.randn(32, 512) * torch.bernoulli(torch.full((32, 512), 0.01)) * 10,
}

print("=== Online vs Offline FP8 Quantization Quality ===")
print(f"Layer: {512} -> {256}\n")
print(f"{'Input Type':<25s} | {'FP16 norm':>10s} | {'Offline err':>12s} | {'Online err':>12s}")
print("-" * 65)

for name, x in test_inputs.items():
    fp16_out = layer.forward_fp16(x)
    
    # Offline: use calibration-derived scale (assume we know the ideal scale)
    calib_scale = x.abs().max() / layer.fp8_max
    offline_out = layer.forward_fp8_offline(x, calib_scale)
    
    # Online: compute scale dynamically
    online_out = layer.forward_fp8_online(x)
    
    offline_err = (fp16_out - offline_out).abs().mean() / fp16_out.abs().mean()
    online_err = (fp16_out - online_out).abs().mean() / fp16_out.abs().mean()
    
    print(f"{name:<25s} | {fp16_out.abs().mean():>10.4f} | {offline_err:>11.4%} | {online_err:>11.4%}")

## 4. FP8 KV-Cache Quantization

### 4.1 Memory Impact

KV-cache is often the largest memory consumer during inference:

```
KV-cache size per token per layer:
  = 2 (K + V) * n_heads * head_dim * bytes_per_element

Example: Llama-3-70B
  n_heads = 64 (for K) + 8 (GQA V groups)
  head_dim = 128
  
  FP16: 2 * (64+8) * 128 * 2 = 36,864 bytes/token/layer
  FP8:  2 * (64+8) * 128 * 1 = 18,432 bytes/token/layer  (2x savings!)

  With 80 layers, 4096 context:
  FP16: 80 * 4096 * 36,864 = ~12 GB per request
  FP8:  80 * 4096 * 18,432 = ~6 GB per request
```

### 4.2 KV-Cache Quantization Strategy

```python
# KV-cache quantization in SGLang

class FP8KVCache:
    """
    Store KV-cache in FP8 format.
    Each token's K and V are quantized per-head.
    """
    def __init__(self, n_layers, max_tokens, n_heads, head_dim):
        # FP8 storage (1 byte per element instead of 2)
        self.k_cache = torch.zeros(
            n_layers, max_tokens, n_heads, head_dim,
            dtype=torch.float8_e4m3fn  # FP8 E4M3
        )
        self.v_cache = torch.zeros_like(self.k_cache)
        
        # Per-head scales (small overhead)
        self.k_scales = torch.ones(n_layers, max_tokens, n_heads, 1)
        self.v_scales = torch.ones(n_layers, max_tokens, n_heads, 1)
    
    def store(self, layer_idx, token_idx, k, v):
        """Store K,V in FP8 format with per-head scaling."""
        # Quantize K
        k_scale = k.abs().amax(dim=-1, keepdim=True) / 448.0
        self.k_cache[layer_idx, token_idx] = (k / k_scale).to(torch.float8_e4m3fn)
        self.k_scales[layer_idx, token_idx] = k_scale
        
        # Quantize V
        v_scale = v.abs().amax(dim=-1, keepdim=True) / 448.0
        self.v_cache[layer_idx, token_idx] = (v / v_scale).to(torch.float8_e4m3fn)
        self.v_scales[layer_idx, token_idx] = v_scale
    
    def load(self, layer_idx, token_indices):
        """Load and dequantize K,V."""
        k_fp8 = self.k_cache[layer_idx, token_indices]
        v_fp8 = self.v_cache[layer_idx, token_indices]
        k = k_fp8.float() * self.k_scales[layer_idx, token_indices]
        v = v_fp8.float() * self.v_scales[layer_idx, token_indices]
        return k, v
```

In [ ]:
# KV-Cache memory savings calculator

def kv_cache_memory(n_layers, max_tokens, n_kv_heads, head_dim, dtype_bytes, 
                    has_scales=False, scale_granularity="per_head"):
    """Calculate KV-cache memory in bytes."""
    # Main KV storage
    kv_size = 2 * n_layers * max_tokens * n_kv_heads * head_dim * dtype_bytes
    
    # Scale storage (if quantized)
    scale_size = 0
    if has_scales:
        if scale_granularity == "per_head":
            scale_size = 2 * n_layers * max_tokens * n_kv_heads * 4  # FP32 scale
        elif scale_granularity == "per_tensor":
            scale_size = 2 * n_layers * 4  # One scale per layer
    
    return kv_size + scale_size

# Model configurations
models = [
    {"name": "Llama-3-8B", "layers": 32, "kv_heads": 8, "head_dim": 128},
    {"name": "Llama-3-70B", "layers": 80, "kv_heads": 8, "head_dim": 128},
    {"name": "Mixtral-8x7B", "layers": 32, "kv_heads": 8, "head_dim": 128},
]

max_tokens = 4096

print("=== KV-Cache Memory: FP16 vs FP8 ===")
print(f"Context length: {max_tokens} tokens\n")
print(f"{'Model':<18s} | {'FP16 (GB)':>10s} | {'FP8 (GB)':>10s} | {'Savings':>8s} | {'Extra reqs':>10s}")
print("-" * 65)

for model in models:
    fp16_mem = kv_cache_memory(model["layers"], max_tokens, model["kv_heads"], 
                               model["head_dim"], dtype_bytes=2)
    fp8_mem = kv_cache_memory(model["layers"], max_tokens, model["kv_heads"],
                              model["head_dim"], dtype_bytes=1, 
                              has_scales=True, scale_granularity="per_head")
    
    fp16_gb = fp16_mem / 1e9
    fp8_gb = fp8_mem / 1e9
    savings = 1 - fp8_gb / fp16_gb
    extra_requests = fp16_gb / fp8_gb - 1  # How many more requests can fit
    
    print(f"{model['name']:<18s} | {fp16_gb:>9.2f} | {fp8_gb:>9.2f} | {savings:>7.1%} | {extra_requests:>9.1%}")

## 5. W8A8 and W4A16 Quantization

### 5.1 W8A8: 8-bit Weights and Activations

Both weights and activations use 8-bit representation. The key challenge is **activation outliers**.

```
Problem: Activations often have outliers in specific channels

Channel values:  [0.1, 0.2, -0.1, 50.0, 0.3, -0.2]  <-- 50.0 is an outlier!
                                     ^^^^
                                  This outlier dominates the scale,
                                  ruining precision for all other values.

SmoothQuant Solution:
  1. Analyze activation distributions offline
  2. Find channels with outliers
  3. "Smooth" by moving difficulty from activations to weights:
     Y = (X * diag(s)^-1) * (diag(s) * W)
     where s = max(|X|)^alpha / max(|W|)^(1-alpha)
```

### 5.2 W4A16: 4-bit Weights, 16-bit Activations

Only weights are quantized to 4 bits. Activations remain in FP16.

**GPTQ (GPT Quantization):**
```python
# GPTQ quantizes weights row-by-row
# Using optimal rounding based on Hessian information
for row in weight_matrix:
    for col in range(n_cols):
        # Find optimal quantized value considering
        # the error it introduces to subsequent columns
        q_val = round_optimal(row[col], hessian_info)
        error = row[col] - q_val
        # Distribute error to remaining columns
        row[col+1:] -= error * hessian_inverse[col, col+1:]
```

**AWQ (Activation-Aware Weight Quantization):**
```python
# AWQ finds per-channel scales that minimize quantization error
# weighted by activation magnitudes (salient channels get more bits)
for channel in range(n_channels):
    activation_importance = activation_stats[channel].mean()
    scale = find_optimal_scale(weight[:, channel], activation_importance)
    weight[:, channel] *= scale  # Scale before quantization
```

In [ ]:
# Compare quantization methods

class QuantizationComparison:
    """Compare different quantization approaches."""
    
    @staticmethod
    def round_to_nearest(x, n_bits, symmetric=True):
        """Simple round-to-nearest quantization."""
        if symmetric:
            qmax = 2 ** (n_bits - 1) - 1
            scale = x.abs().max() / qmax
        else:
            qmin = -(2 ** (n_bits - 1))
            qmax = 2 ** (n_bits - 1) - 1
            scale = (x.max() - x.min()) / (qmax - qmin)
        
        if scale == 0:
            return x, scale
        
        x_q = torch.clamp(torch.round(x / scale), -qmax, qmax)
        x_deq = x_q * scale
        return x_deq, scale
    
    @staticmethod
    def per_channel_quantize(weight, n_bits, axis=0):
        """Per-channel (per-output-channel) quantization."""
        qmax = 2 ** (n_bits - 1) - 1
        scales = weight.abs().amax(dim=1, keepdim=True) / qmax
        scales = torch.clamp(scales, min=1e-10)
        w_q = torch.clamp(torch.round(weight / scales), -qmax, qmax)
        w_deq = w_q * scales
        return w_deq, scales
    
    @staticmethod
    def measure_quality(original, quantized):
        """Measure quantization quality."""
        mse = ((original - quantized) ** 2).mean().item()
        snr = 10 * np.log10((original ** 2).mean().item() / max(mse, 1e-10))
        max_err = (original - quantized).abs().max().item()
        rel_err = ((original - quantized).abs() / (original.abs() + 1e-10)).mean().item()
        return {"mse": mse, "snr_db": snr, "max_err": max_err, "rel_err": rel_err}

# Generate realistic weight matrix
torch.manual_seed(42)
W = torch.randn(256, 512) * 0.02  # Typical LLM weight scale

# Add some structure (outlier channels)
W[:, 10] *= 5  # One channel with larger values
W[:, 200] *= 3

comp = QuantizationComparison()

methods = {
    "FP8 (8-bit)": lambda w: comp.round_to_nearest(w, 8),
    "INT8 per-tensor": lambda w: comp.round_to_nearest(w, 8),
    "INT8 per-channel": lambda w: comp.per_channel_quantize(w, 8),
    "INT4 per-tensor": lambda w: comp.round_to_nearest(w, 4),
    "INT4 per-channel": lambda w: comp.per_channel_quantize(w, 4),
}

print("=== Quantization Method Comparison ===")
print(f"Weight shape: {W.shape}, with outlier channels\n")
print(f"{'Method':<22s} | {'MSE':>12s} | {'SNR (dB)':>10s} | {'Max Error':>10s} | {'Rel Error':>10s}")
print("-" * 75)

for name, quant_fn in methods.items():
    W_q, _ = quant_fn(W)
    quality = comp.measure_quality(W, W_q)
    print(f"{name:<22s} | {quality['mse']:>12.8f} | {quality['snr_db']:>9.1f} | "
          f"{quality['max_err']:>10.6f} | {quality['rel_err']:>9.4%}")

## 6. SGLang Source Code Walkthrough

### 6.1 Quantization Configuration

```python
# Launch SGLang with quantization
# python -m sglang.launch_server \
#     --model meta-llama/Llama-3-8B \
#     --quantization fp8             # FP8 quantization

# Supported quantization methods:
# --quantization fp8          # FP8 E4M3 (online)
# --quantization gptq         # GPTQ 4-bit
# --quantization awq          # AWQ 4-bit  
# --quantization squeezellm   # SqueezeLLM
# --quantization marlin       # Marlin (fast W4A16 kernel)
```

### 6.2 FP8 Linear Layer

```python
# From sglang/srt/layers/quantization/fp8.py

class Fp8LinearMethod:
    """FP8 quantized linear layer."""
    
    def __init__(self, quant_config):
        self.quant_config = quant_config
    
    def create_weights(self, layer, input_size, output_size, ...):
        """Create FP8 weight tensors."""
        # Weights stored in FP8
        layer.weight = Parameter(
            torch.empty(output_size, input_size, dtype=torch.float8_e4m3fn)
        )
        # Per-channel weight scales
        layer.weight_scale = Parameter(
            torch.ones(output_size, 1, dtype=torch.float32)
        )
        # Optional: pre-computed activation scale (offline quantization)
        if self.quant_config.is_static:
            layer.input_scale = Parameter(
                torch.ones(1, dtype=torch.float32)
            )
    
    def apply(self, layer, x, bias=None):
        """Forward pass with FP8 computation."""
        # Dynamic activation quantization
        if hasattr(layer, 'input_scale'):
            # Static quantization: use pre-computed scale
            x_scale = layer.input_scale
        else:
            # Dynamic quantization: compute scale on-the-fly
            x_scale = x.abs().amax() / 448.0
        
        # Quantize activations to FP8
        x_fp8 = (x / x_scale).to(torch.float8_e4m3fn)
        
        # FP8 matrix multiply (hardware-accelerated on H100)
        output = torch._scaled_mm(
            x_fp8,
            layer.weight.T,
            scale_a=x_scale,
            scale_b=layer.weight_scale,
            out_dtype=torch.float16,
        )
        
        if bias is not None:
            output += bias
        
        return output
```

### 6.3 Key Files

| File | Purpose |
|------|--------|
| `sglang/srt/layers/quantization/fp8.py` | FP8 quantization support |
| `sglang/srt/layers/quantization/gptq.py` | GPTQ integration |
| `sglang/srt/layers/quantization/awq.py` | AWQ integration |
| `sglang/srt/layers/quantization/base_config.py` | Base quantization config |
| `sglang/srt/layers/linear.py` | Quantized linear layer dispatch |

In [ ]:
# Simulate FP8 matmul performance characteristics

def estimate_matmul_performance(M, N, K, dtype_bytes, tflops):
    """
    Estimate matmul performance.
    
    Args:
        M, N, K: Matrix dimensions (M x K) @ (K x N)
        dtype_bytes: Bytes per element
        tflops: Hardware throughput in TFLOPS
    """
    # Compute
    flops = 2 * M * N * K
    compute_time_s = flops / (tflops * 1e12)
    
    # Memory (read A + B, write C)
    memory_bytes = (M * K + K * N) * dtype_bytes + M * N * 2  # Output in FP16
    
    return {
        "flops": flops,
        "compute_time_us": compute_time_s * 1e6,
        "memory_bytes": memory_bytes,
        "memory_MB": memory_bytes / 1e6,
    }

# H100 specs
H100_FP16_TFLOPS = 989   # FP16 Tensor Core TFLOPS
H100_FP8_TFLOPS = 1979   # FP8 Tensor Core TFLOPS (2x FP16!)
H100_BW_GBS = 3350       # Memory bandwidth GB/s

# Typical LLM matmul sizes
scenarios = [
    {"name": "Prefill (bs=1, seq=2048)", "M": 2048, "N": 4096, "K": 4096},
    {"name": "Decode (bs=32, seq=1)", "M": 32, "N": 4096, "K": 4096},
    {"name": "MLP up (bs=32)", "M": 32, "N": 11008, "K": 4096},
    {"name": "Prefill large (bs=1, seq=8192)", "M": 8192, "N": 4096, "K": 4096},
]

print("=== FP16 vs FP8 Matmul Performance (H100) ===")
print(f"H100 FP16: {H100_FP16_TFLOPS} TFLOPS, FP8: {H100_FP8_TFLOPS} TFLOPS\n")
print(f"{'Scenario':<35s} | {'FP16 (us)':>10s} | {'FP8 (us)':>10s} | {'FP16 mem':>10s} | {'FP8 mem':>10s} | {'Speedup':>8s}")
print("-" * 95)

for s in scenarios:
    fp16 = estimate_matmul_performance(s["M"], s["N"], s["K"], 2, H100_FP16_TFLOPS)
    fp8 = estimate_matmul_performance(s["M"], s["N"], s["K"], 1, H100_FP8_TFLOPS)
    
    # For small matmuls, memory bandwidth is the bottleneck
    fp16_mem_time = fp16["memory_bytes"] / (H100_BW_GBS * 1e9) * 1e6
    fp8_mem_time = fp8["memory_bytes"] / (H100_BW_GBS * 1e9) * 1e6
    
    fp16_time = max(fp16["compute_time_us"], fp16_mem_time)
    fp8_time = max(fp8["compute_time_us"], fp8_mem_time)
    
    speedup = fp16_time / fp8_time
    
    print(f"{s['name']:<35s} | {fp16_time:>9.1f} | {fp8_time:>9.1f} | "
          f"{fp16['memory_MB']:>9.2f} | {fp8['memory_MB']:>9.2f} | {speedup:>7.2f}x")

## 7. Performance Benchmarks

### 7.1 Throughput Comparison

Typical benchmark results on H100:

| Model | Method | Throughput (tok/s) | Quality (MMLU) | Memory |
|-------|--------|-------------------|----------------|--------|
| Llama-3-8B | FP16 | 100 | 65.3% | 16 GB |
| Llama-3-8B | FP8 | 150 | 65.1% | 8 GB |
| Llama-3-8B | GPTQ-4bit | 180 | 64.5% | 5 GB |
| Llama-3-8B | AWQ-4bit | 175 | 64.8% | 5 GB |
| Llama-3-70B | FP16 | 40 | 79.5% | 140 GB |
| Llama-3-70B | FP8 | 65 | 79.3% | 70 GB |
| Llama-3-70B | GPTQ-4bit | 85 | 78.8% | 35 GB |

*Note: These are representative numbers and vary by hardware and configuration.*

### 7.2 When to Use Each Method

| Scenario | Recommendation | Why |
|----------|---------------|-----|
| Maximum quality | FP16/BF16 | No quantization error |
| Quality-first, some memory savings | FP8 | Minimal quality loss, 2x memory |
| Memory-constrained | GPTQ/AWQ 4-bit | 4x memory savings |
| Throughput-first | FP8 on H100 | Hardware FP8 tensor cores |
| Batch inference | W4A16 + large batches | Best throughput/$ |

In [ ]:
# Comprehensive quantization analysis tool

class QuantizationAnalyzer:
    """Analyze trade-offs of different quantization methods."""
    
    def __init__(self, model_params_B, hidden_dim, n_layers, n_heads, head_dim):
        self.model_params_B = model_params_B
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.head_dim = head_dim
    
    def analyze(self, method, context_length=4096, batch_size=32):
        """Analyze a quantization method."""
        configs = {
            "FP16": {"weight_bits": 16, "act_bits": 16, "kv_bits": 16},
            "FP8": {"weight_bits": 8, "act_bits": 8, "kv_bits": 16},
            "FP8+KV8": {"weight_bits": 8, "act_bits": 8, "kv_bits": 8},
            "GPTQ-4bit": {"weight_bits": 4, "act_bits": 16, "kv_bits": 16},
            "AWQ-4bit": {"weight_bits": 4, "act_bits": 16, "kv_bits": 16},
            "W4A8": {"weight_bits": 4, "act_bits": 8, "kv_bits": 8},
        }
        
        if method not in configs:
            raise ValueError(f"Unknown method: {method}")
        
        cfg = configs[method]
        
        # Model weight memory
        weight_mem_GB = self.model_params_B * 1e9 * (cfg["weight_bits"] / 8) / 1e9
        
        # KV-cache memory per request
        kv_per_token = 2 * self.n_layers * self.n_heads * self.head_dim * (cfg["kv_bits"] / 8)
        kv_per_request_GB = kv_per_token * context_length / 1e9
        kv_total_GB = kv_per_request_GB * batch_size
        
        total_GB = weight_mem_GB + kv_total_GB
        
        return {
            "method": method,
            "weight_mem_GB": weight_mem_GB,
            "kv_per_request_GB": kv_per_request_GB,
            "kv_total_GB": kv_total_GB,
            "total_GB": total_GB,
            "config": cfg,
        }

# Analyze Llama-3-70B
analyzer = QuantizationAnalyzer(
    model_params_B=70, hidden_dim=8192, n_layers=80, 
    n_heads=64, head_dim=128
)

print("=== Llama-3-70B Quantization Analysis ===")
print(f"Context: 4096 tokens, Batch: 32 concurrent requests\n")

methods = ["FP16", "FP8", "FP8+KV8", "GPTQ-4bit", "AWQ-4bit", "W4A8"]
print(f"{'Method':<15s} | {'Weights':>10s} | {'KV/req':>10s} | {'KV total':>10s} | {'Total':>10s} | {'GPUs (80GB)':>11s}")
print("-" * 75)

for method in methods:
    result = analyzer.analyze(method)
    n_gpus = max(1, int(np.ceil(result["total_GB"] / 80)))
    print(f"{method:<15s} | {result['weight_mem_GB']:>9.1f}G | {result['kv_per_request_GB']:>9.2f}G | "
          f"{result['kv_total_GB']:>9.1f}G | {result['total_GB']:>9.1f}G | {n_gpus:>11d}")

In [ ]:
# Visualize memory breakdown
if HAS_PLT:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    methods_data = []
    for method in methods:
        result = analyzer.analyze(method)
        methods_data.append(result)
    
    x = range(len(methods))
    weight_mems = [d["weight_mem_GB"] for d in methods_data]
    kv_mems = [d["kv_total_GB"] for d in methods_data]
    
    ax.bar(x, weight_mems, label='Weights', color='#3498db')
    ax.bar(x, kv_mems, bottom=weight_mems, label='KV-Cache (32 reqs)', color='#e74c3c')
    
    # Add GPU capacity lines
    ax.axhline(y=80, color='green', linestyle='--', alpha=0.7, label='1x A100-80GB')
    ax.axhline(y=160, color='orange', linestyle='--', alpha=0.7, label='2x A100-80GB')
    
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=45, ha='right')
    ax.set_ylabel('Memory (GB)')
    ax.set_title('Llama-3-70B: Memory Breakdown by Quantization Method')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

## 8. Serving with FP8 in SGLang

### 8.1 Configuration

```bash
# Serve with FP8 quantization
python -m sglang.launch_server \
    --model meta-llama/Llama-3-8B-Instruct \
    --quantization fp8 \
    --kv-cache-dtype fp8_e4m3  # Optional: FP8 KV-cache

# Serve with GPTQ model
python -m sglang.launch_server \
    --model TheBloke/Llama-2-70B-GPTQ \
    --quantization gptq \
    --tp 4  # 4-GPU tensor parallel

# Serve with AWQ model
python -m sglang.launch_server \
    --model TheBloke/Llama-2-70B-AWQ \
    --quantization awq \
    --tp 2
```

### 8.2 Quality Verification

Always verify quantized model quality before deployment:

```python
# Quick quality check
import sglang as sgl

# Compare FP16 vs FP8 outputs
prompts = [
    "Explain quantum computing in simple terms:",
    "Write a Python function to sort a list:",
    "What is the capital of France?",
]

for prompt in prompts:
    response = sgl.gen(prompt, max_tokens=100)
    print(f"Prompt: {prompt[:50]}...")
    print(f"Response: {response[:100]}...\n")
```

In [ ]:
# Quality comparison simulation

def simulate_model_quality(weight_matrix, input_data, quantization_method):
    """
    Simulate how quantization affects model output quality.
    Uses cosine similarity as a proxy for generation quality.
    """
    # FP16 reference output
    fp16_output = input_data @ weight_matrix.T
    
    # Quantized output
    comp = QuantizationComparison()
    
    if quantization_method == "FP8":
        w_q, _ = comp.round_to_nearest(weight_matrix, 8)
    elif quantization_method == "INT8_per_channel":
        w_q, _ = comp.per_channel_quantize(weight_matrix, 8)
    elif quantization_method == "INT4_per_tensor":
        w_q, _ = comp.round_to_nearest(weight_matrix, 4)
    elif quantization_method == "INT4_per_channel":
        w_q, _ = comp.per_channel_quantize(weight_matrix, 4)
    else:
        w_q = weight_matrix
    
    q_output = input_data @ w_q.T
    
    # Compute quality metrics
    cos_sim = torch.nn.functional.cosine_similarity(
        fp16_output.flatten().unsqueeze(0), 
        q_output.flatten().unsqueeze(0)
    ).item()
    
    mse = ((fp16_output - q_output) ** 2).mean().item()
    
    return {
        "cosine_similarity": cos_sim,
        "mse": mse,
        "output_norm_ratio": q_output.norm().item() / fp16_output.norm().item(),
    }

# Multi-layer simulation
torch.manual_seed(42)
n_layers = 8
hidden_dim = 512

# Create weight matrices for multiple layers
weights = [torch.randn(hidden_dim, hidden_dim) * 0.02 for _ in range(n_layers)]

# Input data
x = torch.randn(32, hidden_dim)

methods_to_test = ["FP16", "FP8", "INT8_per_channel", "INT4_per_tensor", "INT4_per_channel"]

print("=== Multi-Layer Quantization Quality ===")
print(f"Layers: {n_layers}, Hidden: {hidden_dim}, Batch: 32\n")
print(f"{'Method':<22s} | {'Cos Sim':>10s} | {'MSE':>12s} | {'Norm Ratio':>12s}")
print("-" * 60)

for method in methods_to_test:
    # Propagate through all layers
    total_cos_sim = 0
    total_mse = 0
    total_norm = 0
    
    for layer_w in weights:
        result = simulate_model_quality(layer_w, x, method)
        total_cos_sim += result["cosine_similarity"]
        total_mse += result["mse"]
        total_norm += result["output_norm_ratio"]
    
    avg_cos = total_cos_sim / n_layers
    avg_mse = total_mse / n_layers
    avg_norm = total_norm / n_layers
    
    print(f"{method:<22s} | {avg_cos:>10.6f} | {avg_mse:>12.8f} | {avg_norm:>12.6f}")

## 9. Summary

### Key Takeaways

1. **FP8 E4M3** is the preferred format for inference -- it provides better precision than E5M2 while the reduced range is acceptable with proper scaling.

2. **Per-channel scaling** significantly improves quantization quality over per-tensor, especially when channels have different value ranges.

3. **FP8 KV-cache** provides 2x memory savings, enabling 2x more concurrent requests or longer context lengths.

4. **W4A16 methods** (GPTQ, AWQ) offer 4x memory compression with ~1% quality loss, ideal for memory-constrained deployments.

5. **H100 FP8 tensor cores** deliver 2x compute throughput over FP16, making FP8 a performance win (not just memory saving).

6. **Online quantization** is simpler to deploy (no calibration) but offline quantization gives slightly better quality.

7. Always **benchmark quality** on your specific task before deploying quantized models.

## Exercises

### Exercise 1: Compare Quantization Methods

Implement and compare:
1. Per-tensor FP8 vs per-channel FP8
2. Symmetric vs asymmetric INT8 quantization
3. Measure the output quality degradation through N layers of quantized matmuls

### Exercise 2: Implement SmoothQuant

Implement the SmoothQuant algorithm that migrates quantization difficulty from activations to weights.

### Exercise 3: KV-Cache Quantization Analysis

For a given model:
1. Compute memory savings from FP8 KV-cache
2. Measure the attention quality with quantized KV-cache
3. Find the optimal quantization granularity (per-head, per-token, per-group)

### Exercise 4: Serve a Model with FP8

If you have GPU access:
1. Launch SGLang with `--quantization fp8`
2. Compare throughput and latency vs FP16
3. Run a quality benchmark on a standard eval set

In [ ]:
# Exercise 2 Starter Code: SmoothQuant

def smooth_quant(weight, activation_stats, alpha=0.5):
    """
    TODO: Implement SmoothQuant.
    
    Args:
        weight: [out_features, in_features] weight matrix
        activation_stats: [in_features] per-channel activation max values
        alpha: balance factor (0 = all on weights, 1 = all on activations)
    
    Returns:
        smooth_weight: modified weight matrix
        smooth_scale: scaling factors to apply to activations
    
    Formula:
        s_j = max(|X_j|)^alpha / max(|W_j|)^(1-alpha)
        X_smooth = X * diag(s)^(-1)
        W_smooth = W * diag(s)
    """
    # YOUR CODE HERE
    in_features = weight.shape[1]
    
    # Compute per-channel weight max
    weight_max = weight.abs().amax(dim=0)  # [in_features]
    
    # Compute smoothing scale
    # s = act_max^alpha / weight_max^(1-alpha)
    s = (activation_stats ** alpha) / (weight_max ** (1 - alpha) + 1e-10)
    
    # Apply smoothing
    smooth_weight = weight * s.unsqueeze(0)  # Scale up weights
    smooth_scale = 1.0 / s  # Scale down activations
    
    return smooth_weight, smooth_scale

# Test
torch.manual_seed(42)
W = torch.randn(256, 512) * 0.02

# Simulate activation statistics (some channels have outliers)
act_stats = torch.ones(512) * 0.1
act_stats[10] = 5.0   # Outlier channel
act_stats[200] = 3.0  # Another outlier

smooth_W, smooth_s = smooth_quant(W, act_stats, alpha=0.5)

print("SmoothQuant applied:")
print(f"  Original weight range: [{W.min():.4f}, {W.max():.4f}]")
print(f"  Smooth weight range:   [{smooth_W.min():.4f}, {smooth_W.max():.4f}]")
print(f"  Activation scale range: [{smooth_s.min():.4f}, {smooth_s.max():.4f}]")
print(f"\n  Outlier channel 10: act_stat={act_stats[10]:.1f}, scale={smooth_s[10]:.4f}")
print(f"  Normal channel 50: act_stat={act_stats[50]:.1f}, scale={smooth_s[50]:.4f}")